In [2]:
import torch
from torch import nn
from torch.nn import functional as f
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split

from torchmetrics.classification import Accuracy, Recall


In [4]:
import random
import os
import numpy as np
import torch
from tensorflow import keras

def set_seed(seed=42):
    # 1. 固定 Python 內建的隨機性
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. 固定 NumPy 的隨機性
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)

set_seed(616) 
print("ok")

ok


In [54]:
df = pd.read_csv('/kaggle/input/datasets/organizations/uciml/breast-cancer-wisconsin-data/data.csv')
df.head()

df = df.drop(['id', 'Unnamed: 32'], axis=1)

mapping = {"M": 1, "B": 0}
df["diagnosis"] = df["diagnosis"].replace(mapping)
print(df.head())

df['diagnosis'].value_counts()

s_scaler = StandardScaler()
m_scaler = MinMaxScaler()

x = df.drop(columns = "diagnosis")
y = df["diagnosis"]

x = s_scaler.fit_transform(x)
x = m_scaler.fit_transform(x)

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

   diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0          1        17.99         10.38          122.80     1001.0   
1          1        20.57         17.77          132.90     1326.0   
2          1        19.69         21.25          130.00     1203.0   
3          1        11.42         20.38           77.58      386.1   
4          1        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   symmetry_mean  ...  radius_worst  texture_worst  perimeter_worst  \
0         0.2419  ...         25.38          

/tmp/ipykernel_58/2293831513.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["diagnosis"] = df["diagnosis"].replace(mapping)


In [55]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=621)

In [56]:
x.shape

torch.Size([569, 30])

In [40]:
class VAE(nn.Module):

    def __init__(self, input_dim=30, hidden_dim=32, latent_dim=4):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = f.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = f.relu(self.fc2(z))
        return f.sigmoid(self.fc3(h)) 

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar

def vae_loss(recon_x, x, mu, logvar, kl_weight=0.01, beta = 20.0):
    recon_loss = f.binary_cross_entropy(recon_x, x, reduction="mean")
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    total_loss = recon_loss + beta * (kl_weight * kl_loss)
    return total_loss, recon_loss, kl_loss

vae_dataset = TensorDataset(x_train)
vae_loader = DataLoader(vae_dataset, batch_size=64, shuffle=True)

VAE_model = VAE()
optimizer = Adam(VAE_model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    VAE_model.train()
    total_VAEloss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    
    kl_weight = min(1.0, epoch / 50.0) * 0.1 

    for batch in vae_loader:
        x = batch[0]
        
        optimizer.zero_grad()
        recon, mu, logvar = VAE_model(x)
        VAEloss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, kl_weight, beta = 20.0)

        VAEloss.backward()
        optimizer.step()
        
        total_VAEloss += VAEloss.item() * len(x)
        total_recon_loss += recon_loss.item() * len(x)
        total_kl_loss += kl_loss.item() * len(x)

    total_samples = len(x_train)
    avg_loss = total_VAEloss / total_samples
    avg_recon_loss = total_recon_loss / total_samples
    avg_kl_loss = total_kl_loss / total_samples

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f} | "
            f"Recon: {avg_recon_loss:.4f} | "
            f"KL: {avg_kl_loss:.4f} (wt: {kl_weight:.3f})"
        )


Epoch [1/100] Loss: 0.7066 | Recon: 0.7066 | KL: 0.0115 (wt: 0.000)
Epoch [10/100] Loss: 0.5951 | Recon: 0.5914 | KL: 0.0102 (wt: 0.018)
Epoch [20/100] Loss: 0.5457 | Recon: 0.5437 | KL: 0.0027 (wt: 0.038)
Epoch [30/100] Loss: 0.5339 | Recon: 0.5325 | KL: 0.0012 (wt: 0.058)
Epoch [40/100] Loss: 0.5329 | Recon: 0.5321 | KL: 0.0005 (wt: 0.078)
Epoch [50/100] Loss: 0.5281 | Recon: 0.5274 | KL: 0.0004 (wt: 0.098)
Epoch [60/100] Loss: 0.5272 | Recon: 0.5267 | KL: 0.0002 (wt: 0.100)
Epoch [70/100] Loss: 0.5252 | Recon: 0.5248 | KL: 0.0002 (wt: 0.100)
Epoch [80/100] Loss: 0.5254 | Recon: 0.5252 | KL: 0.0001 (wt: 0.100)
Epoch [90/100] Loss: 0.5248 | Recon: 0.5246 | KL: 0.0001 (wt: 0.100)
Epoch [100/100] Loss: 0.5247 | Recon: 0.5245 | KL: 0.0001 (wt: 0.100)


In [52]:
import torch

VAE_model.eval()

with torch.no_grad():
    
    x_real_label_0 = x_train[y_train == 0]
    

    mu_real, logvar_real = VAE_model.encode(x_real_label_0)
    
    num_samples_to_generate = 80
    base_mu = mu_real[:num_samples_to_generate]
    base_logvar = logvar_real[:num_samples_to_generate]
    
    std = torch.exp(0.5 * base_logvar)
    epsilon = torch.randn_like(std)  # 隨機標準常態噪聲
    z_sampled = base_mu + epsilon * std  # 調製後的隱向量 (z)
    
    generated_features = VAE_model.decode(z_sampled).detach()


generated_labels = torch.zeros(num_samples_to_generate, dtype=y_train.dtype, device=y_train.device)

x_train_augmented = torch.cat((x_train, generated_features), dim=0)
y_train_augmented = torch.cat((y_train, generated_labels), dim=0)

shuffle_indices = torch.randperm(x_train_augmented.size(0))
x_train_sampling = x_train_augmented[shuffle_indices]
y_train_sampling = y_train_augmented[shuffle_indices]


In [57]:
VAE_model.eval()
with torch.no_grad():
    mu, logvar = VAE_model.encode(x_train) 
    recon_x = VAE_model.decode(mu)

recon_error = torch.abs(x_train - recon_x)

x_train_with_VAE = torch.cat((x_train, recon_error), dim=1) 
y_train_with_VAE = y_train.clone()

perm_with_vae = torch.randperm(x_train_with_VAE.size(0))
x_train_with_VAE = x_train_with_VAE[perm_with_vae]
y_train_with_VAE = y_train_with_VAE[perm_with_vae]


In [58]:
class ResBlock(nn.Module):
    def __init__(self, features = 512, bottleneck_features = 64):
        super().__init__()

        self.bn1 = nn.BatchNorm1d(features)
        self.fnc1 = nn.Linear(features, bottleneck_features)
        
        self.bn2 = nn.BatchNorm1d(bottleneck_features)
        self.fnc2 = nn.Linear(bottleneck_features, bottleneck_features)

        self.bn3 = nn.BatchNorm1d(bottleneck_features)
        self.fnc3 = nn.Linear(bottleneck_features, features)
        
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = f.relu(self.bn1(x))
        out = self.dropout(self.fnc1(out))
        
        out = f.relu(self.bn2(out))
        out = self.fnc2(out)
        
        out = f.relu(self.bn3(out))
        out = self.fnc3(out)
        
        return x + out

class BranchBlock(nn.Module):
    def __init__(self, features = 512, depth = 3):
        super().__init__()
        self.res_layers = nn.ModuleList([
            ResBlock(features = features) for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.res_layers:
            x = layer(x)
        return x
class ElementWeight(nn.Module):
    def __init__(self, features = 512):
        super().__init__()
        self.fnc1 = nn.Linear(features, features)

    def forward(self, x):
        weights = torch.sigmoid(self.fnc1(x))

        return x * weights

class Detection_model(nn.Module):
    def __init__(self,dim = 22, num_resBlocks = 5, num_branchs = 5):
        super().__init__()
        self.input = nn.Linear(dim, 512)
        self.bn_in = nn.BatchNorm1d(512)
        
        self.fnc1 = nn.Linear(512, 512)
        self.bn512_1 = nn.BatchNorm1d(512)

        self.parallel_branchs = nn.ModuleList([
            BranchBlock(features = 512, depth = num_resBlocks) for _ in range(num_branchs)
        ])

        self.weights = ElementWeight(features = 512)
                
        self.fnc2 = nn.Linear(512, 512)
        self.bn512_2 = nn.BatchNorm1d(512)
        self.output = nn.Linear(512, 1)

    def forward(self, x):
        x = f.relu(self.bn_in(self.input(x)))
        x = f.relu(self.bn512_1(self.fnc1(x)))

        parallel = sum(blocks(x) for blocks in self.parallel_branchs)
        parallel = self.weights(parallel)
        x = x + parallel 

        x = f.relu(self.bn512_2(self.fnc2(x)))
        x = self.output(x)

        return x

    def prediction(self, x):
        return self.forward(x)

class NN(nn.Module):
    def __init__(self, dim = 11):
        super().__init__()
        self.input = nn.Linear(dim,512)
        self.fnc1 = nn.Linear(512, 512)
        self.output = nn.Linear(512, 1)
        self.bn512_1 = nn.BatchNorm1d(512)
        self.bn512_2 = nn.BatchNorm1d(512)


    def forward(self, x):
        print("ok")
        x = f.relu(self.bn512_1(self.input(x)))
        x = f.relu(self.bn512_2(self.fnc1(x)))
        x = self.output(x)

        return x

print("ok")

ok


In [59]:
training_data = x_train
target = y_train
dim = 30

train_dataset = TensorDataset(training_data, target)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

detection_model = Detection_model(dim = dim, num_resBlocks = 5, num_branchs = 5)


optim = AdamW(detection_model.parameters(), lr = 0.0001)

scheduler = ReduceLROnPlateau(optim, mode='min', factor=0.3, patience=10)

accuracy = Accuracy(task = "binary")
recall = Recall(task="binary")

epochs = 150

pos_weight = torch.tensor([0.8])

for epoch in range(epochs):
    detection_model.train()
    total_loss = 0
    
    epoch_preds = []
    epoch_targets = []

    for batch_x, batch_y in train_loader:
        optim.zero_grad()
        
        y_pred_logits = detection_model(batch_x)
        
        y_pred_logits = y_pred_logits.squeeze() 
        batch_y = batch_y.squeeze()

        loss = f.binary_cross_entropy_with_logits(y_pred_logits, batch_y, pos_weight = pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(detection_model.parameters(), max_norm=1.0)
        optim.step()

        total_loss += loss.item() * len(batch_y)
        
        y_pred_prob = torch.sigmoid(y_pred_logits)
        epoch_preds.append(y_pred_prob.detach())
        epoch_targets.append(batch_y.detach())
        
    avg_loss = total_loss / len(training_data)

    
    scheduler.step(avg_loss)
    
    all_preds = torch.cat(epoch_preds)
    all_targets = torch.cat(epoch_targets)
    
    acc_score = accuracy(all_preds, all_targets)
    recall_score = recall(all_preds, all_targets)
    
    if (epoch+1) % 10 == 0:
        
        current_lr = optim.param_groups[0]['lr']
        print(f"epoch:[{epoch+1}/{epochs}]  loss: {avg_loss:.4f}  acc: {acc_score:.4f}  recall:{recall_score:.4f}  lr:{current_lr:.10f}") 


epoch:[10/150]  loss: 0.0271  acc: 0.9978  recall:0.9941  lr:0.0001000000
epoch:[20/150]  loss: 0.0257  acc: 0.9978  recall:1.0000  lr:0.0001000000
epoch:[30/150]  loss: 0.0055  acc: 1.0000  recall:1.0000  lr:0.0001000000
epoch:[40/150]  loss: 0.0043  acc: 1.0000  recall:1.0000  lr:0.0001000000
epoch:[50/150]  loss: 0.0038  acc: 1.0000  recall:1.0000  lr:0.0001000000
epoch:[60/150]  loss: 0.0162  acc: 0.9978  recall:1.0000  lr:0.0001000000
epoch:[70/150]  loss: 0.0015  acc: 1.0000  recall:1.0000  lr:0.0000300000
epoch:[80/150]  loss: 0.0018  acc: 1.0000  recall:1.0000  lr:0.0000090000
epoch:[90/150]  loss: 0.0011  acc: 1.0000  recall:1.0000  lr:0.0000090000
epoch:[100/150]  loss: 0.0080  acc: 0.9978  recall:0.9941  lr:0.0000090000
epoch:[110/150]  loss: 0.0013  acc: 1.0000  recall:1.0000  lr:0.0000027000
epoch:[120/150]  loss: 0.0033  acc: 0.9978  recall:0.9941  lr:0.0000008100
epoch:[130/150]  loss: 0.0015  acc: 1.0000  recall:1.0000  lr:0.0000002430
epoch:[140/150]  loss: 0.0014  acc

In [60]:

VAE_model.eval()
with torch.no_grad():
    mu, logvar = VAE_model.encode(x_test) 
    recon_x = VAE_model.decode(mu)
        
recon_error = torch.abs(x_test - recon_x)
    
x_test_with_VAE = torch.cat((x_test, recon_error), dim=1) 
y_test_with_VAE = y_train.clone()

perm_with_vae = torch.randperm(x_test_with_VAE.size(0))
x_test_with_VAE = x_test_with_VAE[perm_with_vae]
y_test_with_VAE = y_test_with_VAE[perm_with_vae]


In [62]:
dataset = TensorDataset(x_test, y_test)
loader = DataLoader(dataset, batch_size=16, shuffle=False) 

all_preds = []
all_targets = []

detection_model.eval()

with torch.no_grad():
    for batch_x, batch_y in loader:
        
        y_logits = detection_model(batch_x)
        prob = torch.sigmoid(y_logits)
        
        y_pred = (prob >= 0.5).int()
        
        all_preds.append(y_pred.view(-1))
        all_targets.append(batch_y.view(-1))

final_preds = torch.cat(all_preds, dim=0)
final_targets = torch.cat(all_targets, dim=0).int()

accuracy.reset()
recall.reset()
metric_acc = accuracy(final_preds, final_targets)
metric_recall = recall(final_preds, final_targets)

print(f"Accuracy: {acc_score.item():.4f}")
print(f"Recall:   {recall_score.item():.4f}")

print("\n預測的前 50 筆結果:")
print(final_preds[:50].cpu().numpy()) 

Accuracy: 1.0000
Recall:   1.0000

預測的前 50 筆結果:
[0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 1 0 1 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0
 1 0 0 1 0 1 0 0 0 1 0 0 0]
